In [ ]:
#  Remove conflicting packages
!pip uninstall -y numpy faiss-cpu scikit-learn scipy sentence-transformers transformers
!rm -rf ~/.cache/pip

!pip install --no-cache-dir numpy==1.26.4
!pip install --no-cache-dir faiss-cpu scikit-learn scipy
!pip install --no-cache-dir transformers accelerate bitsandbytes sentencepiece
!pip install --no-cache-dir sentence-transformers tqdm joblib
!pip install --no-cache-dir scispacy
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
print("✓ All dependencies installed")

In [ ]:
import os, json, re, time, pickle, random, sys
import numpy as np
import torch
import torch.nn as nn
from datetime import datetime
from typing import Optional
import requests
import time

from huggingface_hub import login
login("secret_key")

UMLS_API_KEY = "secret_key"

from google.colab import drive
drive.mount('/content/drive')

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from tqdm.notebook import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── PATHS — adjust these to match your Drive layout ──────────────────────────
ENSEMBLE_PATH  = "/content/drive/MyDrive/weighted_ensemble.pth"

# ── Vector DB is loaded from Drive (built locally with build_vector_db_umls_final.py) ──
DRIVE_DATA_DIR = "/content/drive/MyDrive/first_aid_db_4"   # <- folder you uploaded to Drive
DATA_DIR       = "/content/drive/MyDrive/first_aid_db_4"                  # local working copy
os.makedirs(DATA_DIR, exist_ok=True)

# ── MODEL CONFIG ──────────────────────────────────────────────────────────────
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
HF_TOKEN       = ""     # <- paste your HuggingFace token here if model is gated



# ── RAG CONFIG ────────────────────────────────────────────────────────────────
EMBED_MODEL = "sentence-transformers/all-mpnet-base-v2"
RERANK_MODEL     = "cross-encoder/ms-marco-MiniLM-L-6-v2"
MAX_PER_TOPIC    = 500
RETRIEVAL_THRESHOLD = 0.70
#ENTREZ_EMAIL     = "your_email@example.com"   # <- required by NCBI

# ── SEEDS ─────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✓ Config ready | device={device}")
print(f"  LLM           : {LLM_MODEL_NAME}")
print(f"  Ensemble path : {ENSEMBLE_PATH}")
print(f"  Drive data dir: {DRIVE_DATA_DIR}")
print(f"  Local data dir: {DATA_DIR}")

In [ ]:
# ── Copy vector DB from Drive to local /content (faster I/O than Drive) ──────
import shutil

FILES_TO_COPY = ["faiss.index", "bm25.pkl", "metadata.pkl", "chunks.json"]

for fname in FILES_TO_COPY:
    src  = f"{DRIVE_DATA_DIR}/{fname}"
    dst  = f"{DATA_DIR}/{fname}"
    if not os.path.exists(dst):
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"✓ Copied {fname}")
        else:
            print(f"⚠ NOT FOUND in Drive: {src}")
    else:
        print(f"  Already exists locally: {fname}")

print("\n✓ Vector DB files ready in local /content")

In [ ]:
import joblib
class SoftmaxClassifier(nn.Module):
    def __init__(self, p=0.50):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(p * 0.7),

            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(p * 0.5),

            nn.Linear(16, 2),
        )
    def forward(self, x): return self.net(x)


class AttentionClassifier(nn.Module):
    def __init__(self, p=0.65):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2048, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p),

            nn.Linear(64, 2),
        )
    def forward(self, x): return self.net(x)


class FFNClassifier(nn.Module):
    def __init__(self, p=0.70):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(14336, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p),

            nn.Linear(64, 2),
        )
    def forward(self, x): return self.net(x)


class EnsembleClassifier(nn.Module):
    def __init__(self, clf_softmax, clf_attn, clf_ffn,
                 w_softmax=0.2, w_attn=0.3, w_ffn=0.5):
        super().__init__()
        self.clf_softmax = clf_softmax
        self.clf_attn    = clf_attn
        self.clf_ffn     = clf_ffn
        self.w_softmax   = w_softmax
        self.w_attn      = w_attn
        self.w_ffn       = w_ffn

    def forward(self, x_softmax, x_attn, x_ffn):

        p_s = torch.softmax(self.clf_softmax(x_softmax), dim=1)[:, 1].cpu().numpy()
        p_a = torch.softmax(self.clf_attn(x_attn),       dim=1)[:, 1].cpu().numpy()
        p_f = torch.softmax(self.clf_ffn(x_ffn),         dim=1)[:, 1].cpu().numpy()

    #  APPLY CALIBRATION
        p_s = cal_sm.predict(p_s)
        p_a = cal_att.predict(p_a)
        p_f = cal_ffn.predict(p_f)

    # back to tensor
        device = x_softmax.device
        p_s = torch.tensor(p_s, device=device)
        p_a = torch.tensor(p_a, device=device)
        p_f = torch.tensor(p_f, device=device)

        return (
            self.w_softmax * p_s +
            self.w_attn    * p_a +
            self.w_ffn     * p_f
    )

    def predict(self, x_softmax, x_attn, x_ffn):
        self.eval()
        with torch.no_grad():
            prob = self.forward(x_softmax, x_attn, x_ffn)

        prob_val = prob[0].item()

        return prob_val > self.threshold, prob_val


# ── Load scalers from training data (needed to normalise live signals) ────────


scalers = joblib.load("/content/drive/MyDrive/weighted_scalers.pkl")

scaler_softmax = scalers["softmax"]
scaler_attn    = scalers["attn"]
scaler_ffn     = scalers["ffn"]

cal_sm  = scalers["cal_sm"]
cal_att = scalers["cal_att"]
cal_ffn = scalers["cal_ffn"]

print(" Scalers loaded")

# ── Load ensemble checkpoint ──────────────────────────────────────────────────
def load_ensemble(checkpoint_path: str):
      # or ckpt["threshold"]
    clf_s = SoftmaxClassifier().to(device)
    clf_a = AttentionClassifier().to(device)
    clf_f = FFNClassifier().to(device)

    ckpt    = torch.load(checkpoint_path, map_location=device)
    w = ckpt["ensemble_weights"]

    clf_s.load_state_dict(ckpt["softmax_state"])
    clf_a.load_state_dict(ckpt["attention_state"])
    clf_f.load_state_dict(ckpt["ffn_state"])


    ensemble = EnsembleClassifier(clf_s, clf_a, clf_f,
                                  w_softmax=w["softmax"],
                                  w_attn = w["attention"],  # ✅ correct,
                                  w_ffn=w["ffn"]).to(device)

    ensemble.threshold = 0.41
    ensemble.eval()
    print("Threshold:", ensemble.threshold)
    print(f"[Ensemble] Loaded from {checkpoint_path}")
    print(f"  Weights — softmax:{w['softmax']} attention:{w['attention']} ffn:{w['ffn']}")
    return ensemble


# ── Initialise ────────────────────────────────────────────────────────────────

ensemble = load_ensemble(ENSEMBLE_PATH)
print("✓ Ensemble classifier ready")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── Load LLM ──────────────────────────────────────────────────────────────────
def load_llm(model_name: str, hf_token: str = ""):
    print(f"[LLM] Loading {model_name}...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    kwargs = dict(
        quantization_config=bnb_config,
        device_map="auto",
        output_attentions=True,   # needed for attention signal extraction
        output_hidden_states=True # needed for FFN signal extraction
    )
    if hf_token:
        kwargs["token"] = hf_token

    tokenizer = AutoTokenizer.from_pretrained(model_name,
                    token=hf_token if hf_token else None)
    model     = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"  ✓ LLM ready | dtype={next(model.parameters()).dtype}")
    return tokenizer, model


tokenizer, llm_model = load_llm(LLM_MODEL_NAME, HF_TOKEN)

In [ ]:
# ── Internal signal extractor ─────────────────────────────────────────────────

SOFTMAX_TOPK = 20
ATTN_DIM     = 2048
FFN_DIM      = 14336


def extract_first_token_features(prompt, model, tokenizer, top_k=20):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(model.device)

    input_ids = inputs["input_ids"]
    start_pos = input_ids.shape[1]

    # ── First forward pass — get next token ──────────────────
    with torch.no_grad():
        outputs    = model(**inputs)
        logits     = outputs.logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1).unsqueeze(-1)

    full_input_ids = torch.cat([input_ids, next_token], dim=1)

    # ── Second forward pass — get internals ──────────────────
    with torch.no_grad():
        outputs = model(
            input_ids=full_input_ids,
            output_hidden_states=True,
            output_attentions=True
        )

    # 1. SOFTMAX FEATURES
    logits_first     = outputs.logits[:, start_pos - 1, :]
    probs            = torch.softmax(logits_first, dim=-1)
    topk_vals, _     = torch.topk(probs, k=top_k, dim=-1)
    softmax_features = topk_vals.squeeze(0).float().cpu().numpy()

    # 2. ATTENTION FEATURES
    MAX_INPUT_LEN = 64
    attn_last = outputs.attentions[-1]
    num_heads = attn_last.shape[1]

    attn_row   = attn_last[0, :, start_pos, :start_pos]
    actual_len = min(attn_row.shape[1], MAX_INPUT_LEN)

    attn_fixed = torch.zeros(
        num_heads,
        MAX_INPUT_LEN,
        dtype=torch.float32,
        device=attn_last.device
    )
    attn_fixed[:, :actual_len] = attn_row[:, :actual_len].float()

    attn_padded = attn_fixed.cpu().numpy().flatten()

    # 3. FFN FEATURES
    last_layer        = model.model.layers[-1]
    hidden_before_ffn = outputs.hidden_states[-2][:, start_pos, :]
    normed_hidden     = last_layer.post_attention_layernorm(hidden_before_ffn)

    gate           = last_layer.mlp.gate_proj(normed_hidden)
    up             = last_layer.mlp.up_proj(normed_hidden)
    ffn_activation = torch.nn.functional.silu(gate) * up

    ffn_vector = ffn_activation.squeeze(0).detach().float().cpu().numpy()

    return softmax_features, attn_padded, ffn_vector


# ── Generation + signals (CORRECT VERSION) ─────────────────────────────────────

def generate_with_signals(prompt: str, max_new_tokens: int = 256):

    #  Step 1: Extract signals FIRST (matches training)
    x_s, x_a, x_f = extract_first_token_features(
        prompt, llm_model, tokenizer
    )

    #  Step 2: Generate full answer
    inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,

        )

    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    text    = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    #  Step 3: Scale features (IMPORTANT: wrap in list)
    x_s_scaled = scaler_softmax.transform([x_s]).astype(np.float32)
    x_a_scaled = scaler_attn.transform([x_a]).astype(np.float32)
    x_f_scaled = scaler_ffn.transform([x_f]).astype(np.float32)

    #  Step 4: Convert to tensor
    x_s_t = torch.tensor(x_s_scaled, dtype=torch.float32).to(device)
    x_a_t = torch.tensor(x_a_scaled, dtype=torch.float32).to(device)
    x_f_t = torch.tensor(x_f_scaled, dtype=torch.float32).to(device)

    return text, x_s_t, x_a_t, x_f_t


print("✓ Signal extractor ready (FIXED & COMPATIBLE)")

In [ ]:
import faiss
!pip install rank-bm25
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer , util

# ── Paths ─────────────────────────────────────────────────────────────────────
p_faiss = f"{DATA_DIR}/faiss.index"
p_bm25  = f"{DATA_DIR}/bm25.pkl"
p_meta  = f"{DATA_DIR}/metadata.pkl"
p_chunk = f"{DATA_DIR}/chunks.json"

# ── Load prebuilt vector DB (built locally, uploaded to Drive) ────────────────
# Embedding was done locally — we just load the prebuilt index here.
assert os.path.exists(p_faiss), f"Missing: {p_faiss} — did the Drive copy cell run?"
assert os.path.exists(p_bm25),  f"Missing: {p_bm25}"
assert os.path.exists(p_meta),  f"Missing: {p_meta}"

faiss_index = faiss.read_index(p_faiss)
with open(p_bm25, "rb") as f: bm25 = pickle.load(f)
with open(p_meta, "rb") as f: metadata = pickle.load(f)

with open(p_chunk) as f: chunks = json.load(f)

print(f"✓ FAISS index loaded  : {faiss_index.ntotal} vectors")
print(f"✓ BM25 loaded         : {len(metadata)} docs")
print(f"✓ Chunks loaded       : {len(chunks)}")
print("\n Vector DB ready")


In [ ]:
# ================================
#  UMLS CLEAN EXPANSION (ADD THIS)
# ================================

semantic_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
def umls_hybrid_filter(query, candidates):
    strong = []   # trusted (synonyms)
    weak   = []   # needs semantic filtering

    for term, rel, source in candidates:
        term_clean = term.lower().strip()

        # skip obvious junk
        if len(term_clean.split()) > 4:
            continue

        if rel in {"SY", "PT"}:
            strong.append(term)

        else:
            weak.append((term, source))

    #  apply semantic filter ONLY to weak relations
    weak_filtered = semantic_filter(weak, query)

    # combine
    final = list(dict.fromkeys(strong + weak_filtered))

    return final[:3]


def semantic_filter(terms_with_source, query, threshold_entity=0.78, threshold_intent=0.68):
    if not terms_with_source:
        return []

    source_cache = {}
    term_cache = {}

    #  intent embedding (full query is fine)
    query_emb = semantic_model.encode(query, convert_to_tensor=True)

    scored = []

    for term, source in terms_with_source:
        term_clean = term.lower()

        #  remove obvious junk
        if any(x in term_clean for x in ["+", ":", "/", ";", "[", "]"]):
            continue

        #  source embedding
        if source not in source_cache:
            source_cache[source] = semantic_model.encode(source, convert_to_tensor=True)
        base_emb = source_cache[source]

        #  term embedding
        if term not in term_cache:
            term_cache[term] = semantic_model.encode(term, convert_to_tensor=True)
        t_emb = term_cache[term]

        #  similarities
        sim_entity = util.cos_sim(base_emb, t_emb).item()
        sim_query  = util.cos_sim(query_emb, t_emb).item()

        #  dual condition (THIS is the key)
        if sim_entity >= threshold_entity and sim_query >= threshold_intent:
            score = 0.6 * sim_query + 0.4 * sim_entity
            scored.append((term, score))  # rank by query relevance

    scored.sort(key=lambda x: x[1], reverse=True)
    return [t for t, _ in scored]

def clean_umls_expansion(query, umls_candidates):
    terms = umls_hybrid_filter(query, umls_candidates)
    return terms[:3]

In [ ]:
from sentence_transformers import CrossEncoder

# Load embed + rerank models (may already be loaded above)
if 'embed_model' not in dir():
    print(f"Loading {EMBED_MODEL}...")
    embed_model = SentenceTransformer(EMBED_MODEL)

print(f"Loading {RERANK_MODEL}...")
reranker = CrossEncoder(RERANK_MODEL)
print("✓ Retrieval models ready")

In [ ]:
import spacy

# Load scispaCy model ONCE
nlp = spacy.load("en_core_sci_md")
nlp1 = spacy.load("en_ner_bc5cdr_md")

def spacy_tokenize(text):
    doc = nlp(text)
    return [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct and token.lemma_.strip()
    ]

def extract_medical_terms(query):
    doc = nlp1(query)

    terms = []
    for ent in doc.ents:
        label = ent.label_.lower()

        #  keep only real biomedical entities
        if any(k in label for k in ["chemical", "drug", "disease"]):
            terms.append(ent.text)

    return terms

umls_cache = {}

import requests
import time

def get_umls_synonyms(term):
    url_search = "https://uts-ws.nlm.nih.gov/rest/search/current"

    try:
        # Step 1: search → get CUI
        res = requests.get(url_search, params={
            "string": term,
            "apiKey": UMLS_API_KEY
        }, timeout=5).json()

        results = res.get("result", {}).get("results", [])
        if not results:
            return []

        cui = results[0]["ui"]   # take top concept

        # Step 2: fetch relations
        url_rel = f"https://uts-ws.nlm.nih.gov/rest/content/current/CUI/{cui}/relations"

        rel_res = requests.get(url_rel, params={
            "apiKey": UMLS_API_KEY
        }, timeout=5).json()

        rels = rel_res.get("result", [])

        synonyms = []

        for r in rels:

            rel_type = r.get("relationLabel", "")
            name = r.get("relatedIdName", "")

            if name:
                synonyms.append((name, rel_type))

        time.sleep(0.1)
        return synonyms

    except Exception as e:
        print(f"[UMLS Error] {e}")
        return []


def filter_synonyms(synonyms):
    seen = set()
    filtered = []

    for term, rel in synonyms:
        term_clean = term.lower().strip()
        word_count = len(term_clean.split())

        if 1 <= word_count <= 4 and term_clean not in seen:
            seen.add(term_clean)
            filtered.append((term, rel))   # keep tuple!

    return filtered[:5]

In [ ]:
def expand_query_umls(query):
    terms = extract_medical_terms(query)

    if not terms:
        return query

    all_candidates = []

    # collect ALL synonyms
    for term in terms:
        syns = get_umls_synonyms(term)
        syns = filter_synonyms(syns)

        for term_candidate, rel in syns:
            all_candidates.append((term_candidate, rel, term))

    # 🔥 APPLY CLEANING HERE
    clean_terms = clean_umls_expansion(query, all_candidates)

    print("  [UMLS] Raw:", all_candidates)
    print("  [UMLS] Clean:", clean_terms)

    if not clean_terms:
        return query

    return (query)

In [ ]:
#funtions for chunk diversity and deduplication


import numpy as np

def mmr(query, chunks, embed_model, top_k=10, lambda_param=0.7):
    if not chunks:
        return []

    texts = [c["text"] for c in chunks]

    # embeddings
    doc_embeddings = embed_model.encode(texts, normalize_embeddings=True)
    query_embedding = embed_model.encode([query], normalize_embeddings=True)[0]

    selected = []
    selected_indices = []

    # similarity query ↔ docs
    sim_to_query = np.dot(doc_embeddings, query_embedding)

    # pick first (best relevance)
    first_idx = np.argmax(sim_to_query)
    selected_indices.append(first_idx)

    while len(selected_indices) < min(top_k, len(chunks)):
        mmr_scores = []

        for i in range(len(chunks)):
            if i in selected_indices:
                continue

            relevance = sim_to_query[i]

            diversity = max(
                np.dot(doc_embeddings[i], doc_embeddings[j])
                for j in selected_indices
            )

            score = lambda_param * relevance - (1 - lambda_param) * diversity
            mmr_scores.append((i, score))

        next_idx = max(mmr_scores, key=lambda x: x[1])[0]
        selected_indices.append(next_idx)

    return [chunks[i] for i in selected_indices]





In [ ]:
def score_chunks_by_sentences(query, chunks, reranker, max_sentences=10):
    scored_chunks = []

    for c in chunks:
        # split sentences (better to use spaCy ideally)
        sentences = [sent.text.strip() for sent in nlp(c["text"]).sents]


        if not sentences:
            continue

        # CrossEncoder scoring
        pairs = [(query, s) for s in sentences]
        scores = np.array(reranker.predict(pairs))

#  STEP 1: sigmoid normalization (BEST FIX)
        scores = 1 / (1 + np.exp(-scores))   # now in (0,1)

#  STEP 2: aggregate
        topk_score = float(np.mean(np.sort(scores)[-2:]))

        new_c = dict(c)
        new_c["sentence_score"] = topk_score
        scored_chunks.append(new_c)  # or topk_score


    return sorted(scored_chunks, key=lambda x: x["sentence_score"], reverse=True)





In [ ]:


HEADERS = {
    "User-Agent": "MedicalRAGBot/1.0 (mouli@gmail.com)"
}

def wiki_search_titles(query, top_k=3):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    }

    r = requests.get(url, params=params, headers=HEADERS)

    try:
        data = r.json()
    except:
        print("[WIKI ERROR] JSON failed")
        return []

    results = data.get("query", {}).get("search", [])

    if not results:
        return []

    titles = [r["title"] for r in results[:5]]

    #  semantic ranking
    q_emb = semantic_model.encode(query, convert_to_tensor=True)
    t_embs = semantic_model.encode(titles, convert_to_tensor=True)

    sims = util.cos_sim(q_emb, t_embs)[0]

    ranked = sorted(
        zip(titles, sims),
        key=lambda x: x[1],
        reverse=True
    )

    #  take top_k
    selected = [t for t, s in ranked[:top_k] if float(s) > 0.3]

    print(f"[WIKI] Selected titles: {selected}")

    return selected


def wiki_get_full_content(title):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": True,
        "titles": title,
        "format": "json"
    }

    r = requests.get(url, params=params, headers=HEADERS)

    data = r.json()
    pages = data.get("query", {}).get("pages", {})
    page = next(iter(pages.values()))

    return page.get("extract", "")


def wiki_section_search(query):
    print("\n[WIKI] Query:", query)

    titles = wiki_search_titles(query, top_k=3)

    if not titles:
        print("[WIKI] No titles found")
        return None

    combined_text = ""
    urls = []

    for title in titles:
        text = wiki_get_full_content(title)

        if text and len(text) > 100:
            print(f"[WIKI] Using: {title} | len={len(text)}")
            combined_text += text + "\n\n"
            urls.append(f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}")

    if not combined_text:
        return None

    return {
        "title": " + ".join(titles),
        "text": combined_text,
        "url": urls
    }

def wiki_to_chunks(query, top_k=5):
    res = wiki_section_search(query)

    if not res or not res["text"] or len(res["text"]) < 100:
        return []

    #  Split into sentences
    #  Word-based chunking (FIXED SIZE)
    words = res["text"].split()

    chunk_size = 200   # ~200 words per chunk
    overlap = 50       # overlap for continuity

    wiki_chunks = []
    i = 0
    chunk_id = 0

    while i < len(words):
        chunk_words = words[i:i + chunk_size]
        chunk_text = " ".join(chunk_words).strip()

    # keep only meaningful chunks
        if len(chunk_words) > 40:
            wiki_chunks.append({
            "chunk_id": f"wiki_{chunk_id}",
            "text": chunk_text,
            "title": res["title"],
            "source": "wikipedia",
            "url": res["url"][0] if isinstance(res["url"], list) else res["url"],
            "section": "Wiki",
            "entities": []
        })
            chunk_id += 1

        i += (chunk_size - overlap)

    if not wiki_chunks:
        return []

    # 🔥 SAME pipeline as web/pubmed
    wiki_candidates = hybrid_web_retrieve(query, wiki_chunks, top_k=15)
    wiki_reranked = rerank(query, wiki_candidates, top_k=8)
    wiki_diverse = mmr(query, wiki_reranked, embed_model, top_k=5, lambda_param=0.75)
    wiki_scored = score_chunks_by_sentences(query, wiki_diverse, reranker)

    # Normalize scores

    max_rrf = max(c.get("rrf_score", 0) for c in wiki_scored)

    if max_rrf > 0:
        for c in wiki_scored:
            c["rrf_norm"] = c.get("rrf_score", 0) / max_rrf
    else:
        for c in wiki_scored:
            c["rrf_norm"] = 0

    for c in wiki_scored:
        c["final_score"] = 0.75 * c["sentence_score"] + 0.25 * c["rrf_norm"]

    WIKI_CHUNK_THRESHOLD = 0.65   #  tune this (0.3–0.4 range)

    sorted_chunks = sorted(wiki_scored, key=lambda x: x["final_score"], reverse=True)

# filter weak chunks
    filtered = [c for c in sorted_chunks if c["final_score"] >= WIKI_CHUNK_THRESHOLD]

    if not filtered:
        print("[WIKI] All chunks below threshold → rejected")
        return []

    return filtered[:2]  #  return BEST ONLY (consistent with your design)

In [ ]:
HEADERS = {
    "User-Agent": "MedicalRAGBot/1.0 (moulighosh2882003@gmail.com)"
}




def pmc_search(query, top_k=5):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

    params = {
        "db": "pmc",
        "term": query,
        "retmode": "json",
        "retmax": top_k
    }

    try:
        r = requests.get(url, params=params, headers=HEADERS)
        data = r.json()

        ids = data.get("esearchresult", {}).get("idlist", [])

        print("[PMC] IDs:", ids)
        return ids

    except Exception as e:
        print("[PMC Search Error]", e)
        return []



def pmc_fetch_details(id_list):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

    params = {
        "db": "pmc",
        "id": ",".join(id_list),
        "retmode": "xml"
    }

    try:
        r = requests.get(url, params=params, headers=HEADERS)
        xml_text = r.text

        print("[PMC] XML length:", len(xml_text))
        return xml_text

    except Exception as e:
        print("[PMC Fetch Error]", e)
        return ""

import xml.etree.ElementTree as ET

def parse_pmc(xml_text):
    if not xml_text:
        return []

    try:
        root = ET.fromstring(xml_text)
    except Exception as e:
        print("[PMC XML Parse Error]", e)
        return []

    papers = []

    for article in root.findall(".//article"):
        try:
            # 🔹 Title
            title_elem = article.find(".//article-title")
            title = "".join(title_elem.itertext()) if title_elem is not None else ""

            # 🔹 Abstract (preferred)
            abstract_parts = article.findall(".//abstract//p")
            abstract_text = " ".join(
                ["".join(p.itertext()) for p in abstract_parts if p.text]
            )

            if abstract_text and len(abstract_text.split()) > 15:
                text = abstract_text
                source_type = "abstract"

            else:
                #  Fallback → limited full-text
                body_paragraphs = article.findall(".//body//p")

                body_texts = []
                for p in body_paragraphs[:20]:  # LIMIT IS IMPORTANT
                    txt = "".join(p.itertext())
                    if txt:
                        body_texts.append(txt)

                text = " ".join(body_texts)
                source_type = "full_text"

            # 🔹 Final filter
            if text and len(text.split()) > 30:
                papers.append({
                    "title": title,
                    "text": text,
                    "source": "pmc",
                    "source_type": source_type
                })

        except Exception as e:
            print("[PMC Article Parse Error]", e)

    print("[PMC] Papers extracted:", len(papers))
    return papers

def pmc_sentence_pipeline(query):
    ids = pmc_search(query)

    if not ids:
        return []

    xml_text = pmc_fetch_details(ids)
    papers = parse_pmc(xml_text)

    if not papers:
        return []


    q_emb = semantic_model.encode(query, convert_to_tensor=True)

    filtered_papers = []

    for p in papers:
        title_emb = semantic_model.encode(p["title"], convert_to_tensor=True)
        sim = util.cos_sim(q_emb, title_emb).item()

        if sim > 0.20:
            filtered_papers.append(p)

    papers = filtered_papers

#  safety check
    if not papers:
        return []

    all_sentences = []

    for p in papers:
        sentences = [sent.text.strip() for sent in nlp(p["text"]).sents]

        for s in sentences:
            if len(s.split()) > 5:
                all_sentences.append({
                    "text": s,
                    "title": p["title"],
                    "source": "pmc",
                    "source_type": p["source_type"]   # abstract/full_text
                })

    if not all_sentences:
        return []

    #  LIMIT (VERY IMPORTANT)
    if len(all_sentences) > 100:
        all_sentences = all_sentences[:100]

    #  RERANK
    pairs = [(query, s["text"]) for s in all_sentences]
    scores = reranker.predict(pairs)

    # sigmoid normalization
    scores = 1 / (1 + np.exp(-scores))

    #  BOOST ABSTRACT (important)
    for i, s in enumerate(all_sentences):
        score = scores[i]

        if s["source_type"] == "abstract":
            score *= 1.05   # slight boost

        s["score"] = float(score)

    ranked = sorted(all_sentences, key=lambda x: x["score"], reverse=True)

    best = ranked[:2]
    print(f"[PMC] Top score: {ranked[0]['score']:.3f}" if ranked else "No ranked")

    #  THRESHOLD
    if not best or best[0]["score"] < 0.50:
        return []

    return best


In [ ]:
def weighted_rrf(result_lists, weights, k=60):
    scores = {}

    for results, w in zip(result_lists, weights):
        for rank, idx in enumerate(results):
            scores[idx] = scores.get(idx, 0.0) + w * (1.0 / (k + rank + 1))

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def hybrid_web_retrieve(query, web_chunks, top_k=20):


    if not web_chunks:
        return []

    texts = [c["text"] for c in web_chunks]

    # Dense
    embs = embed_model.encode(texts, normalize_embeddings=True)
    q_emb = embed_model.encode([query], normalize_embeddings=True)[0]
    dense_rank = np.argsort(np.dot(embs, q_emb))[::-1][:top_k]

    # Sparse
    bm25_local = BM25Okapi([spacy_tokenize(t) for t in texts])
    sparse_scores = bm25_local.get_scores(spacy_tokenize(query))
    sparse_rank = np.argsort(sparse_scores)[::-1][:top_k]

    # Fusion
    fused = weighted_rrf([dense_rank, sparse_rank], weights=[0.7, 0.3])

    return [
        {**web_chunks[idx], "rrf_score": score}
        for idx, score in fused[:top_k]
    ]

def dense_retrieve(query, top_k=20):
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    _, idxs = faiss_index.search(q_emb, top_k)
    return idxs[0].tolist()

def sparse_retrieve(query, top_k=20):
    tokens = spacy_tokenize(query)
    scores = bm25.get_scores(tokens)
    return np.argsort(scores)[::-1][:top_k].tolist()

def hybrid_retrieve(query, top_k=20):
    expanded = expand_query_umls(query)
    if expanded != query:
        print("  [UMLS] Query expanded")
        print(f"[Query] Expanded : {expanded}")
    fused = weighted_rrf(
    [
        dense_retrieve(expanded, top_k),
        sparse_retrieve(expanded, top_k)
    ],
    weights=[0.7, 0.3]   #  important
)
    chunks = []
    for idx, score in fused[:top_k]:
        if 0 <= idx < len(metadata):
            c = dict(metadata[idx]); c["rrf_score"] = score; chunks.append(c)
    return chunks

def rerank(query, chunks, top_k=5):
    if not chunks: return []
    scores = np.array(reranker.predict([(query, c["text"]) for c in chunks]))

#  normalize SAME as sentence scoring
    scores = 1 / (1 + np.exp(-scores))

    for c, s in zip(chunks, scores):
        c["rerank_score"] = float(s)
    return sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)[:top_k]

def retrieve(query, top_k=5, use_web=False):  #  disable web

    # Step 1: Large candidate pool
    candidates = hybrid_retrieve(query, top_k=30)
    reranked = rerank(query, candidates, top_k=15)

    # Step 2: Diversity
    diverse = mmr(query, reranked, embed_model, top_k=10, lambda_param=0.75)

    # Step 3: Sentence-aware scoring
    scored_chunks = score_chunks_by_sentences(query, diverse, reranker)


    # Normalize RRF
    max_rrf = max(c.get("rrf_score", 0) for c in scored_chunks)

    if max_rrf > 0:
        for c in scored_chunks:
            c["rrf_norm"] = c.get("rrf_score", 0) / max_rrf
    else:
        for c in scored_chunks:
            c["rrf_norm"] = 0

    # Final score
    for c in scored_chunks:
        c["final_score"] = 0.75 * c["sentence_score"] + 0.25 * c["rrf_norm"]

    # Select top chunks
    final = sorted(scored_chunks, key=lambda x: x["final_score"], reverse=True)[:top_k]

    top_score = max(c["final_score"] for c in final) if final else -999
    print(f"  [Retrieve] {len(final)} chunks | top score: {top_score:.3f}")
    best_chunk = final[:1] if final else []
    #  Use PubMed if strong
    if final and top_score > RETRIEVAL_THRESHOLD:
        return best_chunk, "vector_db"

    #  Wikipedia fallback (REPLACES WEB)
    print("  [Fallback] → Wikipedia")

    wiki_chunks = wiki_to_chunks(query)

    if wiki_chunks:
        return wiki_chunks[:top_k], "wikipedia"

    print("  [Fallback] → PubMed")

    pmc_chunks = pmc_sentence_pipeline(query)

    if pmc_chunks:
        return pmc_chunks, "pmc"
    print(" PMC Failed")

    if top_score < 0.50:
        return [], "abstain"

    # Final safety fallback
    return best_chunk, "vector_db"

In [ ]:
def build_grounded_prompt(query: str, context: str) -> str:
    return f"""Answer the following question in one short sentence using the context below.

RULES:
1. Base your answer strictly on the provided context
2. Do NOT mention "context", "Context 1", or any reference. Do NOT cite sources.
3. Do not list multiple items unless explicitly asked.
5. See the full context thoroughly and answer with context only if it answers the question otherwise use prior knowledge.
6. [IMPORTANT RULE] Do NOT dump irrelevant information, just answer the question cleanly in one sentence (only answer what the question is asking for).



CONTEXT:
{context}

QUESTION: {query}

ANSWER (based on the above context):"""


def format_context(chunks: list) -> str:
    return "\n\n".join([c["text"] for c in chunks])


def llm_generate(prompt: str, max_new_tokens: int = 256) -> str:
    """Generate text from LLM without signal extraction (for grounded re-generation)."""
    inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    with torch.no_grad():
        out = llm_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


def medical_rag_pipeline(
    query:          str,
    top_k:          int   = 5,
    initial_answer: str   = None,
    features:       tuple = None,
    max_new_tokens: int   = 80,
    verbose:        bool  = True,
) -> dict:
    """
    Full integrated pipeline.

    Returns dict:
      answer               : final answer string
      initial_answer       : raw LLM answer before grounding
      is_hallucinating     : bool — did classifier flag the initial answer?
      hallucination_prob   : float — ensemble probability score
      retrieval_source     : 'skipped' | 'pubmed' | 'web_fallback'
      sources              : list of source dicts
      chunks               : retrieved chunks (empty if not triggered)
    """
    sep = "="*65

    # ── Step 1: Initial generation + signal extraction ────────────────────────
    if verbose: print(f"\n{sep}\n[Step 1] Initial generation...")
    initial_prompt = f"Answer the following question in one short sentence \nQuestion: {query}\nAnswer:"
    if initial_answer is None:
        initial_answer, x_s, x_a, x_f = generate_with_signals(initial_prompt)
    else:
        if features is None:
            raise ValueError("Features must be provided when initial_answer is given")
        x_s, x_a, x_f = features


    # ── Step 2: Ensemble classifier gate ─────────────────────────────────────
    if verbose: print(f"\n[Step 2] Running ensemble classifier...")
    is_hall, hall_prob = ensemble.predict(x_s, x_a, x_f)
    if verbose:
        flag = "⚠ HALLUCINATING" if is_hall else "✓ CONFIDENT"
        print(f"  Hallucination probability: {hall_prob:.4f} | {flag}")

    # ── Step 3a: Not hallucinating → return directly ──────────────────────────
    if not is_hall:
        if verbose: print(f"\n[Step 3] Classifier: confident → returning answer directly")
        return {
            "answer":             initial_answer,
            "initial_answer":     initial_answer,
            "is_hallucinating":   False,
            "hallucination_prob": hall_prob,
            "retrieval_source":   "skipped",
            "sources":            [],
            "chunks":             [],
        }

    # ── Step 3b: Hallucinating → RAG pipeline ─────────────────────────────────
    if verbose: print(f"\n[Step 3] Classifier: hallucinating → triggering RAG pipeline")

    chunks, retrieval_source = retrieve(query, top_k=top_k)

    if not chunks:
        if verbose:
            print("  [ABSTAIN] No reliable evidence retrieved")

        return {
            "answer": "ABSTAIN",
            "initial_answer": initial_answer,
            "is_hallucinating": True,
            "hallucination_prob": hall_prob,
            "retrieval_source": "abstain",
            "sources": [],
            "chunks": [],
            "abstained": True
           }
    # ── Step 4: Grounded re-generation ────────────────────────────────────────
    if verbose: print(f"\n[Step 4] Grounded re-generation with {len(chunks)} chunks...")
    context        = format_context(chunks)
    grounded_prompt= build_grounded_prompt(query, context)
    grounded_answer= llm_generate(grounded_prompt, max_new_tokens=max_new_tokens)



    # Build sources list
    sources, seen = [], set()
    for c in chunks:
        src = c.get("source","?")
        if src not in seen:
            seen.add(src)
            sources.append({"source":src,"title":c.get("title","")[:100],
                            "url":c.get("url",""),"year":c.get("year","")})

    return {
        "answer":             grounded_answer,
        "initial_answer":     initial_answer,
        "is_hallucinating":   True,
        "hallucination_prob": hall_prob,
        "retrieval_source":   retrieval_source,
        "sources":            sources,
        "chunks":             chunks,
        "abstained": False
    }


def print_result(result: dict):
    sep = "="*65
    print(f"\n{sep}")
    print("INITIAL ANSWER (before grounding):")
    print(f"  {result['initial_answer'][:200]}...")
    print(f"\n  Hallucination prob : {result['hallucination_prob']:.4f}")
    print(f"  Classifier decision: {'⚠ HALLUCINATING → RAG triggered' if result['is_hallucinating'] else '✓ CONFIDENT → returned directly'}")

    if result["is_hallucinating"]:
        print(f"\n{'-'*65}")
        print("GROUNDED ANSWER:")
        print(result["answer"])
        print(f"\n  Retrieval source   : {result['retrieval_source']}")

        print(f"\n  SOURCES ({len(result['sources'])}):")
        for i, s in enumerate(result["sources"], 1):
            line = f"    [{i}] {s['source']}"
            if s.get("year"):  line += f" ({s['year']})"
            if s.get("title"): line += f" — {s['title'][:60]}"
            print(line)
    print(sep)

print("✓ Integrated pipeline ready")

In [ ]:
from datasets import load_dataset

print("Loading dataset...")

dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards")
train_data = dataset["train"]

eval_subset = train_data.select(range(25000, len(train_data)))

data_eval = []
prev_q = None

for row in eval_subset:
    q = row["input"].strip()
    a = row["output"].strip()

    if q == prev_q:
        continue
    if q and a:
        data_eval.append((q, a))
        prev_q = q

print("Valid eval pairs:", len(data_eval))

In [ ]:
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

sim_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

def compute_similarity(a, b):
    emb = sim_model.encode([a, b], convert_to_tensor=True)
    return util.cos_sim(emb[0], emb[1]).item()



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

nli_name = "roberta-large-mnli"

nli_tokenizer = AutoTokenizer.from_pretrained(nli_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_name).to(device)
nli_model.eval()

def run_nli(premise, hypothesis):
    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)
    labels = ["contradiction", "neutral", "entailment"]

    return labels[torch.argmax(probs).item()]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

JUDGE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)

judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

judge_model.eval()

def llm_judge(query, answer, gt):

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a strict medical evaluator.

You MUST follow these rules:
- Output ONLY one word: "YES" or "NO"
- Do NOT explain
- Do NOT add punctuation
- Do NOT add extra text
<|eot_id|>

<|start_header_id|>user<|end_header_id|>
Question: {query}

Ground Truth: {gt}

Answer: {answer}

Is the answer factually correct?<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
"""

    inputs = judge_tokenizer(prompt, return_tensors="pt").to(judge_model.device)

    with torch.no_grad():
        out = judge_model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )



    text = judge_tokenizer.decode(out[0], skip_special_tokens=True).strip().upper()



    text = text.upper()

    if "YES" in text:
        return True
    elif "NO" in text:
        return False
    else:
        return None

In [ ]:
llm_calls = 0

def classify_answer(sim, nli, query, ans, gt):
    global llm_calls

    #  conflict cases
    if nli == "entailment" and sim < 0.60:
        llm_calls += 1
        res = llm_judge(query, ans, gt)
        return None if res is None else not res

    if nli == "contradiction" and sim > 0.85:
        llm_calls += 1
        res = llm_judge(query, ans, gt)
        return None if res is None else not res

    #  strong NLI
    if nli == "entailment":
        return False

    if nli == "contradiction":
        return True

    #  strong similarity
    if sim >= 0.91:
        return False

    if sim <= 0.40:
        return True

    #  fallback
    llm_calls += 1
    res = llm_judge(query, ans, gt)
    return None if res is None else not res

In [ ]:
import os
import numpy as np
from tqdm import tqdm
#8754
SAVE_PATH = "/content/drive/MyDrive/rag_eval_results.npz"
MAX_SAMPLES = 5001

results = []

if os.path.exists(SAVE_PATH):
    saved = np.load(SAVE_PATH, allow_pickle=True)
    results = list(saved["results"])
    start_idx = int(saved["last_idx"]) + 1 if "last_idx" in saved else len(results)
else:
    results = []
    start_idx = 0

for i in tqdm(range(start_idx, min(MAX_SAMPLES, len(data_eval)))):

    query, gt = data_eval[i]

    try:
        prompt = f"Answer the following question in one short sentence \nQuestion: {query}\nAnswer:"

        #  generate ONCE
        base_answer, x_s, x_a, x_f = generate_with_signals(prompt)

        rag_output = medical_rag_pipeline(
            query,
            initial_answer=base_answer,
            features=(x_s, x_a, x_f),
            verbose=False
        )


        final_answer = rag_output["answer"]
        abstained = (final_answer == "ABSTAIN")

        source = rag_output.get("retrieval_source", "unknown")

        if source != "vector_db":
            print("\n RAG TRIGGERED via:", source)
            print(" ANSWER:\n", final_answer)

        # ---- similarity ----
        sim_base  = compute_similarity(base_answer, gt)
        if abstained:
            sim_final = 0.00
        else:
            sim_final = compute_similarity(final_answer, gt)

        # ---- NLI ----
        nli_base = run_nli(gt, base_answer)

        if abstained:
            nli_final = "abstain"

        elif base_answer.strip() == final_answer.strip():
            nli_final = nli_base

        else:
            nli_final = run_nli(gt, final_answer)
        # ---- labels ----
        base_hall  = classify_answer(sim_base,  nli_base,  query, base_answer,  gt)
        if abstained:
            final_hall = False
        else:
            final_hall = classify_answer(
            sim_final,
            nli_final,
            query,
            final_answer,
            gt
           )
        if base_hall is None or final_hall is None:
            continue


        if abstained:

            if base_hall:
                category = "SAFE ABSTAIN"

            else:
                category = "RUINED BY ABSTAIN"

        else:

            if base_hall and not final_hall:
                category = "FIXED by RAG"

            elif not base_hall and final_hall:
                category = "BROKEN by RAG"

            elif base_hall and final_hall:
                category = "STILL WRONG"

            else:
                category = "ALREADY CORRECT"
        results.append({
        "idx": i,

    #  ADD THESE
        "query": query,
        "ground_truth": gt,
        "base_answer": base_answer,
        "final_answer": final_answer,

    # existing metrics
       "sim_base": sim_base,
       "sim_final": sim_final,
       "base_hall": base_hall,
       "final_hall": final_hall,
       "category": category,
       "abstained": abstained
})

        if i % 20 == 0 or len(results) % 20 == 0:
            np.savez(
            SAVE_PATH,
            results=np.array(results, dtype=object),
            last_idx=i
          )
            print("Checkpoint:", len(results))

    except Exception as e:
        print("Error:", e)
        continue

np.savez(
    SAVE_PATH,
    results=np.array(results, dtype=object),
    last_idx=i
)
print("Final saved")

In [ ]:
sim_base_avg  = np.mean([r["sim_base"] for r in results])
sim_final_avg = np.mean([r["sim_final"] for r in results])
answered_results = [
    r for r in results
    if not r["abstained"]
]

if answered_results:
    selective_similarity = np.mean([
        r["sim_final"]
        for r in answered_results
    ])
else:
    selective_similarity = 0

improvement_rate = np.mean([
    r["sim_final"] > r["sim_base"]
    for r in results
    if not r["abstained"]
])

base_hall = np.sum([r["base_hall"] for r in results])
final_hall = np.sum([
    r["final_hall"]
    for r in results
    if not r["abstained"]
])

abstain_count = np.sum([
    r["abstained"]
    for r in results
])

coverage = (
    len(results) - abstain_count
) / len(results)

hall_reduction = (base_hall - final_hall) / max(base_hall, 1)

categories = {}
for r in results:
    categories[r["category"]] = categories.get(r["category"], 0) + 1

# =========================
# PRINT
# =========================
print("\n===== FINAL RESULTS =====")
print("Baseline Similarity:", sim_base_avg)
print("RAG Similarity:", sim_final_avg)
print("Selective Similarity:", selective_similarity)

print("\nImprovement Rate:", improvement_rate)
print("Hallucination Reduction:", hall_reduction)
print("\nAbstentions:", abstain_count)
print("Coverage:", coverage)
abstention_rate = abstain_count / len(results)
print("Abstention Rate:", abstention_rate)
print("\nCategory Breakdown:")
for k, v in categories.items():
    print(k, ":", v)

print("\nLLM judge used:", llm_calls)


# =========================
# SAVE TO TXT
# =========================
SAVE_TXT = "/content/drive/MyDrive/rag_final_metrics.txt"

with open(SAVE_TXT, "a") as f:   # use "w" if you want overwrite
    f.write("\n" + "="*50 + "\n")
    f.write("FINAL RESULTS\n")
    f.write("="*50 + "\n")

    f.write(f"Baseline Similarity: {sim_base_avg:.6f}\n")
    f.write(f"RAG Similarity:      {sim_final_avg:.6f}\n")
    f.write(f"Selective Similarity:{selective_similarity:.6f}\n")
    f.write(f"Abstentions:             {abstain_count}\n")
    f.write(f"Coverage:                {coverage:.6f}\n")
    f.write(f"Abstention Rate:         {abstention_rate:.6f}\n")

    f.write(f"\nImprovement Rate:        {improvement_rate:.6f}\n")
    f.write(f"Hallucination Reduction: {hall_reduction:.6f}\n")

    f.write("\nCategory Breakdown:\n")
    for k, v in categories.items():
        f.write(f"{k}: {v}\n")

    f.write(f"\nLLM judge used: {llm_calls}\n")